# Cá nhân hoá kết quả — thiết kế 3 tầng

**Nguyên tắc**: tương quan ở cấp population (nhóm nhỏ, nhiều session lặp
lại) KHÔNG tự động đúng cho từng cá nhân. Cá nhân hoá thật sự cần đủ session
của chính người đó, không phải suy diễn thẳng từ trung bình nhóm.

Notebook này dựng báo cáo cá nhân theo 3 tầng, tăng dần yêu cầu về độ tin cậy:

- **Tầng 1 — Ngay sau session**: Δ trạng thái pre/post, `device_confidence`,
  HR/RMSSD trước→trong thiền của chính session đó. Không cần thống kê nhóm,
  luôn hiển thị được.
- **Tầng 2 — Xu hướng cá nhân theo thời gian**: đường xu hướng
  `device_confidence`/RMSSD/alpha power qua các ngày đã tập của CHÍNH họ.
  Chỉ hiển thị nếu participant có **≥5 session hợp lệ** — nếu ít hơn, đánh
  dấu "chưa đủ dữ liệu" thay vì suy diễn.
- **Tầng 3 — Gợi ý tham khảo nhóm**: đối chiếu vị trí cá nhân trên đường xu
  hướng population đã biết (`sleep_hours_roll3` ↔ `mean_alpha_power_rel`), trình bày rõ là thông
  tin tham khảo chung, không phải chẩn đoán cá nhân.

**Không đưa vào cá nhân hoá** (quá kỹ thuật/chưa đủ vững ở cấp cá nhân):
composite EEG-mood score, diễn giải "tỉnh táo/gắn kết" — chỉ dùng nội bộ,
không hiển thị cho end-user.

**Dữ liệu dùng**: hoàn toàn từ `metadata_multimodal/session_metadata_survey_features.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

REPO_ROOT = Path("../..").resolve()
META_PATH = REPO_ROOT / "data" / "analytics" / "metadata_multimodal" / "session_metadata_survey_features.csv"
OUT_DIR = Path(".").resolve() / "outputs"
CARDS_DIR = OUT_DIR / "cards"
OUT_DIR.mkdir(exist_ok=True)
CARDS_DIR.mkdir(exist_ok=True)

metadata = pd.read_csv(META_PATH, low_memory=False)
metadata = metadata.sort_values(["participant_key", "study_day"]).reset_index(drop=True)

STATE_VARS = ["stress", "calmness", "focus", "sleepiness", "bodyTension", "mood"]
MIN_SESSIONS_FOR_TREND = 5

group_a = metadata[metadata["participant_key"].str.startswith("P_A")].copy()
participants = sorted(group_a["participant_key"].unique())
print(f"{len(participants)} participant nhom A (co du lieu thiet bi)")

## Tầng 1 — Snapshot session gần nhất

In [ ]:
def build_tier1(pid):
    """Delta trang thai + device_confidence + HR/RMSSD cua session GAN NHAT."""
    sub = group_a[group_a["participant_key"] == pid].dropna(subset=["study_day"])
    if len(sub) == 0:
        return None
    last = sub.iloc[-1]
    row = {"participant_key": pid, "latest_study_day": last["study_day"]}
    for v in STATE_VARS:
        row[f"delta_{v}"] = last.get(f"delta_{v}")
    row["device_confidence"] = last.get("device_confidence")
    row["hr_before"] = last.get("ppg_hr_mean_bpm__REST_BEFORE")
    row["hr_during"] = last.get("ppg_hr_mean_bpm__MEDITATION")
    row["rmssd_before"] = last.get("ppg_rmssd_ms__REST_BEFORE")
    row["rmssd_during"] = last.get("ppg_rmssd_ms__MEDITATION")
    return row

tier1_rows = [build_tier1(p) for p in participants]
tier1_df = pd.DataFrame([r for r in tier1_rows if r is not None])
tier1_df.to_csv(OUT_DIR / "tier1_latest_session_snapshot.csv", index=False)
out1 = OUT_DIR / 'tier1_latest_session_snapshot.csv'
print(f"Da luu: {out1}")
tier1_df

## Tầng 2 — Xu hướng cá nhân (chỉ tính nếu ≥5 session hợp lệ)

In [ ]:
def build_tier2(pid):
    """Xu huong (Spearman r theo study_day) cho device_confidence, Delta RMSSD, Delta alpha_power_rel."""
    sub = group_a[group_a["participant_key"] == pid].copy()
    sub["rmssd_delta"] = sub["ppg_rmssd_ms__MEDITATION"] - sub["ppg_rmssd_ms__REST_BEFORE"]

    result = {"participant_key": pid}
    targets = {
        "device_confidence": "device_confidence",
        "rmssd_delta": "rmssd_delta",
        "alpha_power_rel_delta": "delta_eeg_mean_alpha_power_rel",
    }
    n_sessions = sub["study_day"].notna().sum()
    result["n_sessions"] = n_sessions
    result["du_du_lieu_tang2"] = bool(n_sessions >= MIN_SESSIONS_FOR_TREND)

    for label, col in targets.items():
        s = sub[["study_day", col]].dropna()
        if len(s) >= 3:
            r, p = stats.spearmanr(s["study_day"], s[col])
            result[f"{label}_r"] = r
            result[f"{label}_p"] = p
            result[f"{label}_n"] = len(s)
        else:
            result[f"{label}_r"] = np.nan
            result[f"{label}_p"] = np.nan
            result[f"{label}_n"] = len(s)
    return result

tier2_rows = [build_tier2(p) for p in participants]
tier2_df = pd.DataFrame(tier2_rows)
tier2_df.to_csv(OUT_DIR / "tier2_personal_trends.csv", index=False)
out2 = OUT_DIR / 'tier2_personal_trends.csv'
n_ok = tier2_df['du_du_lieu_tang2'].sum()
print(f"Da luu: {out2}")
print(f"{n_ok}/{len(tier2_df)} participant du du lieu (>= {MIN_SESSIONS_FOR_TREND} session) de hien thi tang 2")
tier2_df

## Tầng 3 — Vị trí cá nhân trên đường xu hướng nhóm (`sleep_hours_roll3` ↔ `mean_alpha_power_rel`)

Chỉ dùng để **tham khảo**, không phải chẩn đoán cá nhân — luôn hiển thị kèm
toàn bộ điểm dữ liệu nhóm để người dùng tự thấy mức độ phân tán thật.

In [ ]:
pop = metadata[["participant_key", "sleep_hours_roll3", "delta_eeg_mean_alpha_power_rel"]].dropna()
pop_slope, pop_intercept = np.polyfit(pop["sleep_hours_roll3"], pop["delta_eeg_mean_alpha_power_rel"], 1)
pop_r, pop_p = stats.spearmanr(pop["sleep_hours_roll3"], pop["delta_eeg_mean_alpha_power_rel"])
print(f"Duong xu huong nhom: slope={pop_slope:.4f}, r={pop_r:.3f}, p={pop_p:.2e}, n={len(pop)}")

def build_tier3(pid):
    sub = pop[pop["participant_key"] == pid]
    return {"participant_key": pid, "n_diem_ca_nhan": len(sub),
            "sleep_hours_roll3_median": sub["sleep_hours_roll3"].median() if len(sub) else np.nan}

tier3_rows = [build_tier3(p) for p in participants]
tier3_df = pd.DataFrame(tier3_rows)
tier3_df.to_csv(OUT_DIR / "tier3_group_reference.csv", index=False)
tier3_df

## Bảng tổng hợp 3 tầng (1 dòng/participant)

In [ ]:
summary = tier1_df.merge(tier2_df, on="participant_key").merge(tier3_df, on="participant_key")
summary_out = OUT_DIR / "personal_summary_all_participants.csv"
summary.to_csv(summary_out, index=False)
print(f"Da luu: {summary_out} -- {summary.shape}")
summary[["participant_key", "latest_study_day", "device_confidence", "n_sessions", "du_du_lieu_tang2", "n_diem_ca_nhan"]]

## Thẻ cá nhân (personal card) — ví dụ trực quan cho từng participant

3 panel khớp 3 tầng: (1) Δ trạng thái session gần nhất, (2) xu hướng
`device_confidence` theo ngày (nếu đủ dữ liệu), (3) vị trí cá nhân trên
đường xu hướng nhóm sleep→alpha. Lưu 1 file PNG/participant vào `outputs/cards/`.

In [ ]:
def draw_personal_card(pid):
    sub = group_a[group_a["participant_key"] == pid].copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    fig.suptitle(f"Thẻ cá nhân — {pid}", fontsize=13, fontweight="bold")

    # Panel 1: Delta trang thai session gan nhat
    row1 = tier1_df[tier1_df["participant_key"] == pid].iloc[0]
    deltas = [row1.get(f"delta_{v}") for v in STATE_VARS]
    colors = ["#1a9c5a" if (pd.notna(d) and (v in ("calmness", "focus", "mood") and d > 0 or v in ("stress", "sleepiness", "bodyTension") and d < 0)) else "#b4650a" if pd.notna(d) else "#ccc" for v, d in zip(STATE_VARS, deltas)]
    axes[0].bar(STATE_VARS, [d if pd.notna(d) else 0 for d in deltas], color=colors)
    axes[0].axhline(0, color="gray", lw=1)
    latest_day = row1['latest_study_day']
    axes[0].set_title(f"Tầng 1: Δ trạng thái (ngày {latest_day:.0f})", fontsize=10)
    axes[0].tick_params(axis="x", rotation=40, labelsize=8)

    # Panel 2: xu huong device_confidence
    row2 = tier2_df[tier2_df["participant_key"] == pid].iloc[0]
    s = sub[["study_day", "device_confidence"]].dropna()
    if row2["du_du_lieu_tang2"] and len(s) >= 3:
        axes[1].scatter(s["study_day"], s["device_confidence"], color="#2f6fed", alpha=0.7)
        slope, intercept = np.polyfit(s["study_day"], s["device_confidence"], 1)
        xs = np.linspace(s["study_day"].min(), s["study_day"].max(), 20)
        axes[1].plot(xs, slope * xs + intercept, color="#b4650a", lw=2)
        conf_r = row2['device_confidence_r']
        axes[1].set_title(f"Tầng 2: xu hướng relax (r={conf_r:.2f})", fontsize=10)
    else:
        axes[1].text(0.5, 0.5, "Chưa đủ dữ liệu\n(cần ≥5 session)", ha="center", va="center", transform=axes[1].transAxes, fontsize=10, color="gray")
        axes[1].set_title("Tầng 2: xu hướng relax", fontsize=10)
    axes[1].set_xlabel("study_day")
    axes[1].set_ylabel("device_confidence")

    # Panel 3: vi tri ca nhan tren duong xu huong nhom
    axes[2].scatter(pop["sleep_hours_roll3"], pop["delta_eeg_mean_alpha_power_rel"], color="#ccc", alpha=0.4, s=20, label="Toàn nhóm")
    mysub = pop[pop["participant_key"] == pid]
    axes[2].scatter(mysub["sleep_hours_roll3"], mysub["delta_eeg_mean_alpha_power_rel"], color="#2f6fed", s=50, label=pid, zorder=5)
    xs = np.linspace(pop["sleep_hours_roll3"].min(), pop["sleep_hours_roll3"].max(), 20)
    axes[2].plot(xs, pop_slope * xs + pop_intercept, color="#b4650a", lw=2, label="Xu hướng nhóm")
    axes[2].set_title("Tầng 3: tham khảo nhóm (không phải chẩn đoán)", fontsize=10)
    axes[2].set_xlabel("sleep_hours_roll3")
    axes[2].set_ylabel("Δ alpha_power_rel")
    axes[2].legend(fontsize=7)

    fig.tight_layout()
    fig.savefig(CARDS_DIR / f"{pid}_personal_card.png", dpi=120)
    plt.close(fig)

for p in participants:
    draw_personal_card(p)
print(f"Da luu {len(participants)} the ca nhan vao {CARDS_DIR}")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(CARDS_DIR / f"{participants[0]}_personal_card.png")))

## Tổng kết

- `outputs/personal_summary_all_participants.csv` — 1 dòng/participant, đủ
  3 tầng, dùng làm nguồn cho UI cá nhân hoá thật (app/dashboard).
- `outputs/cards/*.png` — thẻ trực quan mẫu, tham khảo bố cục khi thiết kế
  UI thật (không phải sản phẩm cuối).
- Nhắc lại nguyên tắc: **Tầng 1 luôn đúng** (dữ liệu trực tiếp), **Tầng 2
  chỉ đúng khi đủ session cá nhân**, **Tầng 3 luôn phải ghi rõ là tham khảo
  nhóm**, không khẳng định đúng cho riêng từng người.